In [ ]:
"""
=============================================================
 NOTEBOOK 4 — CatBoost Regression Modelling
=============================================================
This notebook trains and evaluates cut-specific CatBoost
regression models to predict beef eating quality (MQ4 score)
from carcass grading data and engineered chill curve features.

MQ4 target is a weighted composite:
  Tenderness (×0.4) + Juicy (×0.3) + Flavour (×0.3)

Recommended target: EQ_component_corrected
  (bias-corrected version produced by 02b_panellist_bias_correction)

Modelling approach:
  - CatBoost selected for native handling of mixed feature
    types without one-hot encoding
  - GroupKFold cross-validation with farm as the grouping key,
    preventing data leakage across cuts from the same carcass
  - Cut-specific models trained for each primal
  - Benchmarked against the MSA industry standard framework

MLflow tracking:
  - All runs logged with hyperparameters and CV metrics
  - Feature importance plots and residual diagnostics
    stored as artefacts per run
  - Trained models versioned in the MLflow model registry

Inputs:
  - Carcass-level CSV with membrane features
    (output of 03_feature_engineering_membrane)

Outputs:
  - Trained cut-specific models (MLflow model registry)
  - Feature importance CSVs per cut
  - Cross-validation performance summary
=============================================================
"""

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import shap
from pathlib import Path
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import KFold, GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## 1. Configuration

In [ ]:
# ── Paths — update to your environment ───────────────────────────────────────
path = 'C:/'

MLFLOW_ROOT = Path(f"{path}ML_flow_APB")
mlflow.set_tracking_uri(f"file:{MLFLOW_ROOT / 'mlruns'}")
mlflow.set_experiment("ABP_EQ_Prediction")

# Input files — carcass-level CSVs with membrane features (from notebook 03)
INPUT_FILES = {
    'LD': f"{path}carcass_level/ALL_LD_carcass_level_with_membrane.csv",
    'SM': f"{path}carcass_level/ALL_SM_carcass_level_with_membrane.csv",
    'FT': f"{path}carcass_level/ALL_FT_carcass_level.csv",
    'GM': f"{path}carcass_level/ALL_GM_carcass_level.csv",
}

OUTPUT_DIR = f"{path}carcass_level/kfold_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Recommended target — bias-corrected EQ from notebook 02b
TARGET_COL = 'EQ_component_corrected'
GROUP_COL  = 'Farm Location Code-Unique Farmer ID'  # GroupKFold grouping
N_SPLITS   = 5

## 2. Column Drop Lists
Post-aging columns and cook lab measurements are excluded —
they are not available at grading time in production.

In [ ]:
# Columns never used as features
NON_FEATURE_COLS = {
    'Eartag', 'split',
    'EQ_mean', 'EQ_corrected_mean',
    'EQ_std', 'EQ_corrected_std',
    'EQ_min', 'EQ_max', 'EQ_range', 'N_panellists',
    'EQ_corrected', 'EQ_component_corrected',
    'EQ_correction_delta',
}

# Core leakage / admin columns — dropped for all cuts
CORE_DROPS = [
    'kill_season', 'kill_year', 'County Code',
    'Keeper Code', 'Herd Number', 'Dam x Sire',
    # Cook lab — not available at grading time
    'total_wt_loss_pct', 'cook_wt_loss_pct', 'thaw_wt_loss_pct',
    'FZN WT', 'RAW WEIGHT', 'COOKED WEIGHT',
    'RAW DEPTH', 'COOKED DEPTH', 'COOK TEMP', 'CKD PH',
    'dept_change', 'dept_increase_flag', 'dept_shrink_pct',
    # Timestamp strings
    '1hr-Time', '2hr-Time', '4hr-Time', '6hr-Time',
    '9hr-Time', '14hr-Time', '19hr-Time',
    # Admin
    'Kill Date', 'Date of Birth (DOB)', 'Lot ID', 'Batch',
    'Factory', 'Primal', 'Ribeye Star Fat',
]

# Cut-specific post-aging drops
CUT_DROPS = {
    'LD': [
        '21days - Loin Steak A - MSA Marbling',
        '21days - Loin Steak B - MSA Marbling',
        '21days - Loin Steak Avg - MSA Marbling',
        '21days -  Loin -  pH', '21days - Loin -  temp',
        'Condusctivity - 21days -  Loin -  With Grain',
        'Conductivity - 21days - Loin - Against grain',
        'Conductivity - 21days - Loin - Cut face',
    ],
    'FT': [
        '21days - Fillet Steak A - MSA Marbling',
        '21days - Fillet Steak B - MSA Marbling',
        '21days - Fillet Steak Avg - MSA Marbling',
        '21days -  Filet -  pH', '21days - Filet -  temp',
        'Conductivity - 21days -   Fillet -  With Grain',
        'Conductivity - 21days -  Fillet -  Against grain',
        'Conductivity - 21days -  Fillet -  Cut face',
    ],
    'GM': [
        '21days - Rump Heart True Steak A - MSA Marbling',
        '21days - Rump Heart True Steak B - MSA Marbling',
        '21days - Rump Heart True Steak Avg - MSA Marbling',
        '21days - Rump Heart Mignon Steak A - MSA Marbling',
        '21days - Rump Heart Mignon Steak B - MSA Marbling',
        '21days - Rump Heart Mignon Steak Avg - MSA Marbling',
        '21days -  Rump Heart True -  pH', '21days -  Rump Heart True - temp',
        '21days -  Rump Heart Mignon - pH', '21days - Rump Heart Mignon - temp',
        'Conductivity - 21days -   Rump Heart True -  With Grain',
        'Conductivity - 21days -  Rump Heart True -  Against grain',
        'Conductivity - 21days -  Rump Heart True -  Cut face',
        'Conductivity - 21days -   Rump Heart Mignon -   With Grain',
        'Conductivity - 21days -  Rump Heart Mignon -   Against grain',
        'Conductivity - 21days -  Rump Heart Mignon -   Cut face',
    ],
    'SM': [
        '21days - Topside Steak A - MSA Marbling',
        '21days - Topside Steak B - MSA Marbling',
        '21days - Topside Steak Avg - MSA Marbling',
        '21days -  Topside -  pH', '21days - Topside -  temp',
        'Conductivity - 21days -Topside -  With Grain',
        'Conductivity - 21days - Topside -  Against grain',
        'Conductivity - 21days - Topside -  Cut face',
    ],
}

## 3. Helper Functions

In [ ]:
# Default CatBoost parameters — tuned for small carcass-level datasets (~400 rows)
DEFAULT_PARAMS = {
    'iterations':             5000,
    'early_stopping_rounds':  100,
    'learning_rate':          0.02,
    'depth':                  4,
    'l2_leaf_reg':            10,
    'min_data_in_leaf':       8,
    'subsample':              0.8,
    'random_seed':            42,
    'loss_function':          'RMSE',
    'eval_metric':            'R2',
    'verbose':                0,
}


def prepare_features(df: pd.DataFrame, cut_code: str):
    """Apply column drops and return (X, feature_cols, cat_feature_cols)."""
    all_drops = CORE_DROPS + CUT_DROPS.get(cut_code, [])
    drop_cols = [c for c in all_drops if c in df.columns]
    df_clean  = df.drop(columns=drop_cols)

    feature_cols = [c for c in df_clean.columns if c not in NON_FEATURE_COLS]
    X = df_clean[feature_cols].copy()

    cat_feature_cols = [c for c in feature_cols if X[c].dtype in ['object', 'string']]
    for c in cat_feature_cols:
        X[c] = X[c].astype(str).fillna('missing')

    print(f"  Dropped : {len(drop_cols)} columns")
    print(f"  Features: {len(feature_cols)} ({len(cat_feature_cols)} categorical)")
    return X, feature_cols, cat_feature_cols


def compute_metrics(y_true, y_pred):
    """Return RMSE, MAE, R²."""
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return {
        'r2':   round(1 - ss_res / ss_tot, 4),
        'rmse': round(np.sqrt(np.mean((y_true - y_pred) ** 2)), 4),
        'mae':  round(np.mean(np.abs(y_true - y_pred)), 4),
    }


def log_shap_artifacts(model, X_explain, feature_cols, cat_features, max_rows=500):
    """Log SHAP summary plot and importance CSV to MLflow."""
    X_s = X_explain.sample(min(len(X_explain), max_rows), random_state=42).copy()
    explain_pool = Pool(X_s, cat_features=cat_features)
    shap_vals  = model.get_feature_importance(explain_pool, type='ShapValues')
    shap_values = shap_vals[:, :-1]

    plt.figure()
    shap.summary_plot(shap_values, X_s[feature_cols], show=False)
    shap_png = 'shap_summary.png'
    plt.savefig(shap_png, bbox_inches='tight', dpi=150)
    plt.close()
    mlflow.log_artifact(shap_png)

    mean_abs  = np.abs(shap_values).mean(axis=0)
    shap_imp  = pd.DataFrame({'feature': feature_cols, 'mean_abs_shap': mean_abs})
    shap_imp  = shap_imp.sort_values('mean_abs_shap', ascending=False)
    shap_csv  = 'shap_importance.csv'
    shap_imp.to_csv(shap_csv, index=False)
    mlflow.log_artifact(shap_csv)

## 4. GroupKFold Cross-Validation with MLflow

In [ ]:
def run_kfold_analysis(
        df, cut_code, target_col='EQ_component_corrected',
        group_col=None, n_splits=5, model_params=None, output_dir=None):
    """
    Run GroupKFold cross-validation and train a final model.

    Fold strategy
    -------------
    GroupKFold on farm ID ensures no farm appears in both train
    and test within any fold — the most honest estimate of
    generalisation to new farms.
    """
    params = {**DEFAULT_PARAMS, **(model_params or {})}

    print(f"\n{'='*58}")
    print(f"  CatBoost GroupKFold Cross Validation")
    print(f"  Cut:     {cut_code}  |  Target: {target_col}")
    print(f"  n={len(df)}  |  folds={n_splits}  |  group={group_col or 'Random KFold'}")
    print(f"{'='*58}")

    if target_col not in df.columns:
        raise ValueError(f"Target '{target_col}' not found. Available EQ cols: "
                         f"{[c for c in df.columns if 'EQ' in c]}")

    X, feature_cols, cat_feature_cols = prepare_features(df.copy(), cut_code)
    y = df[target_col].values
    cat_indices = [X.columns.get_loc(c) for c in cat_feature_cols if c in X.columns]

    if group_col and group_col in df.columns:
        splitter   = GroupKFold(n_splits=n_splits)
        split_iter = splitter.split(X, y, groups=df[group_col].values)
    else:
        splitter   = KFold(n_splits=n_splits, shuffle=True, random_state=42)
        split_iter = splitter.split(X, y)

    print(f"\n  {'Fold':<6} {'N train':>8} {'N test':>8} "
          f"{'R²':>8} {'RMSE':>8} {'MAE':>8} {'Best iter':>10}")
    print(f"  {'─'*54}")

    fold_results   = []
    all_importance = None

    with mlflow.start_run(run_name=f"{cut_code}_{target_col}_kfold") as parent_run:
        mlflow.log_param('cut', cut_code)
        mlflow.log_param('target', target_col)
        mlflow.log_param('n_splits', n_splits)
        mlflow.log_param('group_col', group_col or 'None')
        mlflow.log_param('n_carcasses', len(df))
        mlflow.log_param('n_features', len(feature_cols))
        mlflow.log_params(params)

        # Save feature list as artefact for reproducibility
        feat_path = 'feature_list.json'
        with open(feat_path, 'w') as f:
            json.dump(feature_cols, f, indent=2)
        mlflow.log_artifact(feat_path)

        for fold, (train_idx, test_idx) in enumerate(split_iter, 1):
            X_tr, X_te = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
            y_tr, y_te = y[train_idx], y[test_idx]

            train_pool = Pool(X_tr, label=y_tr, cat_features=cat_indices)
            test_pool  = Pool(X_te, label=y_te, cat_features=cat_indices)

            with mlflow.start_run(run_name=f"fold_{fold}", nested=True):
                model = CatBoostRegressor(**params)
                model.fit(train_pool, eval_set=test_pool)

                preds   = model.predict(X_te)
                metrics = compute_metrics(y_te, preds)

                mlflow.log_metric('r2',   metrics['r2'])
                mlflow.log_metric('rmse', metrics['rmse'])
                mlflow.log_metric('mae',  metrics['mae'])
                try:
                    mlflow.log_metric('best_iteration', model.best_iteration_)
                except Exception:
                    pass

                model_path = f'model_fold_{fold}.cbm'
                model.save_model(model_path)
                mlflow.log_artifact(model_path)

            fi = pd.Series(model.get_feature_importance(), index=feature_cols)
            all_importance = fi if all_importance is None else all_importance + fi

            fold_results.append({'fold': fold, 'n_train': len(X_tr),
                                  'n_test': len(X_te), **metrics,
                                  'best_iteration': model.best_iteration_})
            print(f"  {fold:<6} {len(X_tr):>8} {len(X_te):>8} "
                  f"{metrics['r2']:>8.4f} {metrics['rmse']:>8.4f} "
                  f"{metrics['mae']:>8.4f} {model.best_iteration_:>10}")

        # Aggregate metrics to parent run
        r2s   = [f['r2']   for f in fold_results]
        rmses = [f['rmse'] for f in fold_results]
        maes  = [f['mae']  for f in fold_results]
        summary = {'r2_mean': round(np.mean(r2s), 4), 'r2_std': round(np.std(r2s), 4),
                   'r2_min':  round(np.min(r2s),  4), 'r2_max': round(np.max(r2s), 4),
                   'rmse_mean': round(np.mean(rmses), 4), 'rmse_std': round(np.std(rmses), 4),
                   'mae_mean':  round(np.mean(maes),  4), 'mae_std':  round(np.std(maes), 4)}

        for k, v in summary.items():
            mlflow.log_metric(k, v)

        cv_df = pd.DataFrame(fold_results)
        cv_csv = 'cv_metrics.csv'
        cv_df.to_csv(cv_csv, index=False)
        mlflow.log_artifact(cv_csv)

        # Average feature importance
        avg_importance = (all_importance / n_splits).sort_values(ascending=False)
        importance_df  = avg_importance.reset_index()
        importance_df.columns = ['Feature', 'Importance (%)']
        importance_df['Importance (%)'] = importance_df['Importance (%)'].round(4)

        fi_csv = f'{cut_code}_feature_importance.csv'
        importance_df.to_csv(fi_csv, index=False)
        mlflow.log_artifact(fi_csv)

        print(f"\n  {'─'*54}")
        print(f"  R²  : {summary['r2_mean']:.4f} ± {summary['r2_std']:.4f}  "
              f"[{summary['r2_min']:.4f} – {summary['r2_max']:.4f}]")
        print(f"  RMSE: {summary['rmse_mean']:.4f} ± {summary['rmse_std']:.4f}")
        print(f"  MAE : {summary['mae_mean']:.4f} ± {summary['mae_std']:.4f}")
        print(f"\n  TOP 20 FEATURES (averaged across folds)")
        for _, row in importance_df.head(20).iterrows():
            print(f"  {row['Feature']:<45} {row['Importance (%)']:>6.2f}%")

        # Train final model on all training carcasses
        print(f"\n  Training final model on all TRAIN carcasses...")
        if 'split' in df.columns:
            train_mask = df['split'] == 'train'
            X_final_tr = X[train_mask.values].copy()
            y_final_tr = y[train_mask.values]
            X_final_te = X[~train_mask.values].copy()
            y_final_te = y[~train_mask.values]
        else:
            X_final_tr, y_final_tr = X.copy(), y
            X_final_te, y_final_te = None, None

        final_pool  = Pool(X_final_tr, label=y_final_tr, cat_features=cat_indices)
        final_model = CatBoostRegressor(**{**params, 'verbose': 100})

        if X_final_te is not None and len(X_final_te) > 0:
            eval_pool = Pool(X_final_te, label=y_final_te, cat_features=cat_indices)
            final_model.fit(final_pool, eval_set=eval_pool)
            final_metrics = compute_metrics(y_final_te, final_model.predict(X_final_te))
            print(f"  HELD-OUT TEST  R²={final_metrics['r2']:.4f}  "
                  f"RMSE={final_metrics['rmse']:.4f}  MAE={final_metrics['mae']:.4f}")
            for k, v in final_metrics.items():
                mlflow.log_metric(f'final_holdout_{k}', v)
        else:
            final_model.fit(final_pool)

        final_model_path = f'{cut_code}_final_model.cbm'
        final_model.save_model(final_model_path)
        mlflow.log_artifact(final_model_path)

        # SHAP on holdout set
        if X_final_te is not None and len(X_final_te) > 0:
            log_shap_artifacts(final_model, X_final_te,
                               feature_cols=feature_cols,
                               cat_features=cat_indices)

        if output_dir:
            os.makedirs(output_dir, exist_ok=True)
            final_model.save_model(os.path.join(output_dir, f'{cut_code}_kfold_model.cbm'))
            cv_df.to_csv(os.path.join(output_dir, f'{cut_code}_kfold_results.csv'), index=False)
            importance_df.to_csv(os.path.join(output_dir, f'{cut_code}_feature_importance.csv'), index=False)

    print(f"{'='*58}\n")
    return {'cut': cut_code, 'target': target_col, 'summary': summary,
            'fold_results': fold_results, 'feature_importance': importance_df,
            'final_model': final_model, 'feature_cols': feature_cols}

## 5. Run All Cuts

In [ ]:
def run_all_cuts(input_files, output_dir, target_col='EQ_component_corrected',
                  group_col='Farm Location Code-Unique Farmer ID',
                  n_splits=5, model_params=None):
    """Run k-fold analysis for all four cuts and print a comparison table."""
    all_results = {}
    for cut_code, fp in input_files.items():
        print(f"\n{'#'*58}\n  {cut_code}\n{'#'*58}")
        df  = pd.read_csv(fp, encoding='utf-8-sig')
        grp = group_col if group_col in df.columns else None
        all_results[cut_code] = run_kfold_analysis(
            df=df, cut_code=cut_code, target_col=target_col,
            group_col=grp, n_splits=n_splits,
            model_params=model_params, output_dir=output_dir)

    print(f"\n{'='*65}")
    print(f"  CROSS-CUT SUMMARY  |  Target: {target_col}")
    print(f"{'='*65}")
    print(f"  {'Cut':<10} {'R² mean':>10} {'R² std':>10} {'RMSE mean':>12} {'RMSE std':>10}")
    print(f"  {'─'*55}")
    for code, res in all_results.items():
        s = res['summary']
        print(f"  {code:<10} {s['r2_mean']:>10.4f} {s['r2_std']:>10.4f} "
              f"{s['rmse_mean']:>12.4f} {s['rmse_std']:>10.4f}")
    return all_results


# Run
all_results = run_all_cuts(
    input_files  = INPUT_FILES,
    output_dir   = OUTPUT_DIR,
    target_col   = TARGET_COL,
    group_col    = GROUP_COL,
    n_splits     = N_SPLITS,
)

## 6. Training Curve Visualisation

In [ ]:
# Plot training curves for a chosen cut's final model
CUT = 'LD'  # change as needed
model = all_results[CUT]['final_model']

evals_result = model.get_evals_result()
train_rmse   = evals_result['learn']['RMSE']
test_rmse    = evals_result['validation']['RMSE']
best_iter    = model.get_best_iteration()

plt.figure(figsize=(10, 6))
plt.plot(train_rmse, label='Train RMSE', linewidth=2, color='steelblue')
plt.plot(test_rmse,  label='Test RMSE',  linewidth=2, color='tomato')
plt.axvline(x=best_iter, color='green', linestyle='--', linewidth=1.5,
            label=f'Best Iteration ({best_iter})')
plt.scatter([best_iter], [test_rmse[best_iter]], color='green', s=100, zorder=5)
plt.xlabel('Iteration', fontsize=12)
plt.ylabel('RMSE', fontsize=12)
plt.title(f'CatBoost Training Curves — {CUT}', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{CUT}_training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Best iteration: {best_iter}")
print(f"Best test RMSE: {test_rmse[best_iter]:.4f}")

## 7. Feature Importance Visualisation

In [ ]:
CUT = 'LD'
importance_df = all_results[CUT]['feature_importance']

plt.figure(figsize=(10, 8))
top20 = importance_df.head(20)
plt.barh(top20['Feature'], top20['Importance (%)'], color='steelblue')
plt.xlabel('Importance (%)', fontsize=12)
plt.title(f'Top 20 Features — {CUT} (averaged across folds)', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{CUT}_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()